In [ ]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp

In [ ]:

spark = SparkSession.builder \
    .appName("CDC dimension") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()


In [ ]:
gold_path_cliente = "s3a://analytics/medallion/03-gold/dim_cliente" 

In [ ]:
dim_cliente = spark.read.format("delta").load(gold_path_cliente) 

In [ ]:

# Adiciona timestamps de auditoria
dim_cliente = dim_cliente.withColumn("created_at", current_timestamp()) \
                         .withColumn("updated_at", current_timestamp())

In [ ]:
# -----------------------------
# CDC para dim_cliente
# -----------------------------

In [ ]:

from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp


spark = SparkSession.builder \
    .appName("CDC dimension") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()


gold_path_cliente = "s3a://analytics/medallion/03-gold/dim_cliente" 

dim_cliente = spark.read.format("delta").load(gold_path_cliente) 



# Adiciona timestamps de auditoria
dim_cliente = dim_cliente.withColumn("created_at", current_timestamp()) \
                         .withColumn("updated_at", current_timestamp())


# Função de CDC para a dimensão
def cdc_dim_cliente(df):
    tabela_gold = "gold.dim_cliente"
    chave_merge = "t.id = s.id"

    if not spark.catalog.tableExists(tabela_gold):
        # Cria tabela pela primeira vez
        df.write.format("delta").mode("overwrite").saveAsTable(tabela_gold)
        print(f"Tabela {tabela_gold} criada e dados carregados.")
    else:
        # Tabela já existe → merge (CDC)
        gold_table = DeltaTable.forName(spark, tabela_gold)
        gold_table.alias("t").merge(
            df.alias("s"),
            chave_merge
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
        print(f"Tabela {tabela_gold} atualizada via CDC.")

# Executa o CDC para dim_cliente
cdc_dim_cliente(dim_cliente)


NameError: name 'spark' is not defined